In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np
from bertopic import BERTopic
from sklearn.linear_model import LinearRegression

C:\Users\atliu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Loading Embeddings

In [36]:
embeddings = np.load("embeddings.npy")

df = pd.read_parquet("documents.parquet")

# Model Decision

In [37]:
from sklearn.feature_extraction.text import CountVectorizer
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

vectorizer = CountVectorizer(stop_words="english")
analyzer = vectorizer.build_analyzer()

In [38]:
tokenized_docs  = [analyzer(doc) for doc in df['text'].to_list()]

In [39]:
def compute_coherence(topic_model, docs, sample_size=None, coherence_type="c_v"):
    """
    Faster coherence computation for BERTopic
    """

    # 🔹 Sample kullan (çok önerilir)
    if sample_size is not None:
        docs = docs[:sample_size]

    # 🔹 Tokenize
    vectorizer = CountVectorizer(stop_words="english")
    analyzer = vectorizer.build_analyzer()
    tokenized_docs = [analyzer(doc) for doc in docs]

    # 🔹 Dictionary
    dictionary = Dictionary(tokenized_docs)

    # 🔹 Topic words al
    topics = topic_model.get_topics()
    topic_words = [
        [word for word, _ in topics[topic_id]]
        for topic_id in topics
        if topic_id != -1
    ]

    # 🔹 Coherence hesapla
    coherence_model = CoherenceModel(
        topics=topic_words,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence=coherence_type
    )

    return coherence_model.get_coherence()

In [40]:
docs = df["text"].tolist()

results = []
models = {}
topics_dict = {} 
for min_size in [20, 50, 100]:

    model = BERTopic(
        min_topic_size=min_size,
        calculate_probabilities=False,
        verbose=True
    )

    topics, _ = model.fit_transform(docs, embeddings)

    score = compute_coherence(
        model,
        docs,
        sample_size=5000  # hız için sample
    )
    topic_count = len(model.get_topic_info()) - 1 
    results.append((min_size, score, topic_count))
    models[min_size] = model
    topics_dict[min_size] = topics

print(results)

2026-02-26 12:51:10,240 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-02-26 12:51:46,631 - BERTopic - Dimensionality - Completed ✓
2026-02-26 12:51:46,634 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-02-26 12:52:10,974 - BERTopic - Cluster - Completed ✓
2026-02-26 12:52:11,005 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-02-26 12:52:31,535 - BERTopic - Representation - Completed ✓
2026-02-26 12:53:39,774 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-02-26 12:54:14,916 - BERTopic - Dimensionality - Completed ✓
2026-02-26 12:54:14,919 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-02-26 12:54:23,507 - BERTopic - Cluster - Completed ✓
2026-02-26 12:54:23,528 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-02-26 12:54:45,212 - BERTopic - Representation - Completed ✓
2026-02-26 12:55:20,636 - BERTopic - D

[(20, np.float64(0.44211872540642644), 709), (50, np.float64(0.47064555982948114), 352), (100, np.float64(0.49888628861726275), 209)]


In [41]:
df_results = pd.DataFrame(results, columns=["min_topic_size", "coherence", "topic_count"])
df_results

,min_topic_size,coherence,topic_count
0,20,0.442119,709
1,50,0.470646,352
2,100,0.498886,209


In [42]:
model = models[100]
topics = topics_dict[100]
reduced_model = model.reduce_topics(docs, nr_topics=100)
reduced_topics, _ = reduced_model.transform(docs, embeddings)

2026-02-26 12:57:09,143 - BERTopic - Topic reduction - Reducing number of topics
2026-02-26 12:57:09,220 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-02-26 12:57:29,018 - BERTopic - Representation - Completed ✓
2026-02-26 12:57:29,043 - BERTopic - Topic reduction - Reduced number of topics from 210 to 100
2026-02-26 12:57:33,410 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-02-26 12:57:33,835 - BERTopic - Dimensionality - Completed ✓
2026-02-26 12:57:33,836 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-02-26 12:57:52,365 - BERTopic - Cluster - Completed ✓


In [43]:
#model = BERTopic(
#    min_topic_size=100,
#    calculate_probabilities=False,
#    verbose=True,
#    nr_topics=100)

# Trend Analysis

In [44]:
df["topic"] = reduced_topics
df["month"] = df["published"].dt.to_period("M")

trend = (
    df[df.topic != -1] # -1 means outliers, we ignore them
    .groupby(["month", "topic"])
    .size()
    .unstack(fill_value=0)
)

trend_share = trend.div(trend.sum(axis=1), axis=0)

In [45]:
def compute_trend_scores(trend_share, window=12, min_share=0.001):
    scores = {}

    recent = trend_share.tail(window)
    x = np.arange(len(recent)).reshape(-1, 1)

    mean_share = recent.mean()
    valid_topics = mean_share[mean_share > min_share].index

    for topic in valid_topics:
        y = recent[topic].values

        if y.sum() == 0:
            continue

        model = LinearRegression()
        model.fit(x, y)

        scores[topic] = model.coef_[0]

    return pd.Series(scores).sort_values(ascending=False)

def compute_log_trend_scores(trend_share, window=12, min_share=0.001):
    scores = {}

    recent = trend_share.tail(window)
    x = np.arange(len(recent)).reshape(-1, 1)

    mean_share = recent.mean()
    valid_topics = mean_share[mean_share > min_share].index

    for topic in valid_topics:
        y = recent[topic].values



        y_log = np.log(y + 1e-6)

        model = LinearRegression()
        model.fit(x, y_log)

        scores[topic] = model.coef_[0]

    return pd.Series(scores).sort_values(ascending=False)



In [46]:
log_trend_scores_12 = compute_log_trend_scores(trend_share, window=12)

print("🔥 Emerging topics:")
print(log_trend_scores_12.head(10))

print("\n📉 Declining topics:")
print(log_trend_scores_12.tail(10))

🔥 Emerging topics:
86    0.267703
53    0.104337
10    0.071179
64    0.063735
77    0.051395
57    0.043906
81    0.040272
43    0.038972
82    0.038514
4     0.037054
dtype: float64

📉 Declining topics:
91   -0.268265
72   -0.277814
67   -0.281996
83   -0.284570
78   -0.292512
89   -0.298532
47   -0.315288
80   -0.324845
29   -0.352795
79   -0.418036
dtype: float64


In [47]:
log_trend_scores_24 = compute_log_trend_scores(trend_share, window=24)

print("🔥 Emerging topics:")
print(log_trend_scores_24.head(10))

print("\n📉 Declining topics:")
print(log_trend_scores_24.tail(10))

🔥 Emerging topics:
82    0.121441
4     0.072077
10    0.067816
37    0.049771
53    0.046023
81    0.025033
70    0.024851
41    0.024585
61    0.021590
57    0.020873
dtype: float64

📉 Declining topics:
90   -0.082600
80   -0.088711
94   -0.090038
78   -0.091609
73   -0.093338
67   -0.099764
79   -0.106493
47   -0.112831
29   -0.150379
75   -0.168644
dtype: float64


In [48]:
len(log_trend_scores_12), len(log_trend_scores_24)

(91, 94)

#  Visualization

In [49]:
import plotly.express as px
from sklearn.preprocessing import MinMaxScaler


In [50]:
def clean_label(name):
    parts = name.split("_")[2:]  # topic_x_ kısmını at
    return " ".join(parts[:3])   # ilk 3 keyword yeterli

def generate_label(topic_id, top_n=3):
    words = [word for word, _ in reduced_model.get_topic(topic_id)]
    return " / ".join(words[:top_n])

def assign_quadrant(row):
    if row["slope_24m"] > 0 and row["slope_12m"] > 0:
        return "🚀 Strong & Accelerating"
    
    elif row["slope_24m"] > 0 and row["slope_12m"] < 0:
        return "📈 Strong but Slowing"
    
    elif row["slope_24m"] < 0 and row["slope_12m"] > 0:
        return "🌱 Emerging"
    
    else:
        return "📉 Declining"

In [73]:
topics_df  = reduced_model.get_topic_info()
topics_df = topics_df[topics_df.Topic != -1]

In [74]:
topics_df["label"] = topics_df["Name"].apply(clean_label)
topics_df["label"] = topics_df["label"].str.capitalize()

In [75]:
slope_df =pd.DataFrame({
    "slope_12m": log_trend_scores_12,
    "slope_24m": log_trend_scores_24
})
slope_df["Topic"] = slope_df.index

topics_df = topics_df.merge(slope_df, on="Topic")
topics_df.set_index("Topic", inplace=True)


In [76]:
topic_map = topics_df[["label"]].to_dict()["label"]

In [77]:
topics_df["category"] = topics_df.apply(assign_quadrant, axis=1)
topics_df["acceleration"] = topics_df["slope_12m"] - topics_df["slope_24m"]

In [78]:
topics_df["growth_score"] = (topics_df["slope_12m"] * np.log1p(topics_df["Count"]))
scaler = MinMaxScaler()
topics_df["growth_size"] = scaler.fit_transform(topics_df[["growth_score"]])
topics_df["growth_size"] = topics_df["growth_size"] * 40 + 10

In [82]:
topics_df.head()

,Count,Name,Representation,Representative_Docs,label,slope_12m,slope_24m,category,acceleration,growth_score,growth_size
Topic,,,,,,,,,,,
0,9043,0_policy_learning_reinforcement_driving,"[policy, learning, reinforcement, driving, rl,...",[Language Understanding for Field and Service ...,Learning reinforcement driving,-0.007305,-0.003292,📉 Declining,-0.004012,-0.066545,35.389869
1,7384,1_speech_audio_asr_recognition,"[speech, audio, asr, recognition, music, speak...",[MOSS-Speech: Towards True Speech-to-Speech Mo...,Audio asr recognition,-0.005970,-0.002280,📉 Declining,-0.003690,-0.053180,35.527703
2,4347,2_medical_clinical_biomedical_patient,"[medical, clinical, biomedical, patient, healt...",[Effective Medical Code Prediction via Label I...,Clinical biomedical patient,-0.021419,0.000045,📈 Strong but Slowing,-0.021464,-0.179439,34.225604
3,3799,3_translation_languages_multilingual_language,"[translation, languages, multilingual, languag...",[Attention based Sequence to Sequence Learning...,Languages multilingual language,-0.016363,-0.025941,📉 Declining,0.009578,-0.134877,34.685162
4,3734,4_reasoning_cot_mathematical_llms,"[reasoning, cot, mathematical, llms, models, m...","[Characterizing, Evaluating, and Optimizing Co...",Cot mathematical llms,0.037054,0.072077,🚀 Strong & Accelerating,-0.035022,0.304791,39.219422


In [84]:
topics_df["label"].nunique()

95

In [79]:
topics_df.to_csv("topics_trend_analysis.csv")
trend_share.to_csv("trend_share.csv")


In [58]:
top_topics = topics_df["slope_12m"].sort_values(ascending=False).head(10).index

df_plot = trend_share[top_topics]
df_plot.index = df_plot.index.to_timestamp()
df_plot.columns = df_plot.columns.map(topic_map)

fig = px.line(
    df_plot,
    x=df_plot.index,
    y=df_plot.columns,
    title="Trend Share Over Time 12-Month Slope"
)
fig.update_layout(
    width=1200,
    height=600
)
fig.show()

In [59]:
top_topics = topics_df["slope_24m"].sort_values(ascending=False).head(10).index

df_plot = trend_share[top_topics]
df_plot.index = df_plot.index.to_timestamp()
df_plot.columns = df_plot.columns.map(topic_map)

fig = px.line(
    df_plot,
    x=df_plot.index,
    y=df_plot.columns,
    title="Trend Share Over Time 24-Month Slope"
)
fig.update_layout(
    width=1200,
    height=600
)
fig.show()

In [60]:
fig = px.scatter(
    topics_df,
    x="slope_24m",
    y="slope_12m",
    color="category",
    size="Count",
    hover_name="label",

    title="AI-Powered Research Trend Quadrant"
)

fig.add_vline(x=0, line_dash="dash")
fig.add_hline(y=0, line_dash="dash")

fig.update_layout(
    xaxis_title="24M Growth Rate",
    yaxis_title="12M Growth Rate",
    template="plotly_white"
)

fig.show()

In [61]:
top_topics = topics_df[(topics_df["slope_12m"] > 0)].sort_values(by="acceleration", ascending=False).head(10).index
topics_df = topics_df.loc[top_topics]
    
fig = px.scatter(
    topics_df,
    x="slope_24m",
    y="slope_12m",
    color="category",
    size="Count",
    hover_name="label",
    title="AI-Powered Research Trend Quadrant",
)


fig.add_vline(x=0, line_dash="dash")
fig.add_hline(y=0, line_dash="dash")

fig.update_layout(
    xaxis_title="24M Growth Rate",
    yaxis_title="12M Growth Rate",
    template="plotly_white"
)

fig.show()

In [62]:
df_plot = topics_df.dropna()
df_plot["category"] = df_plot.apply(assign_quadrant, axis=1)
fig = px.scatter(
    df_plot,
    x="slope_24m",
    y="slope_12m",
    size="growth_size",
    color="category",
    hover_name="label",
    title="AI-Powered Research Trend Intelligence Quadrant"
)

fig.add_vline(x=0, line_dash="dash")
fig.add_hline(y=0, line_dash="dash")

fig.update_layout(template="plotly_white")

fig.show()

In [63]:
top10_growth = (topics_df.sort_values("growth_score", ascending=False).head(10).index)

topics_df_plot = topics_df.loc[top10_growth]
fig = px.scatter(
    topics_df_plot,
    x="slope_24m",
    y="slope_12m",
    size="growth_size",
    color="category",
    hover_name="label",
    title="AI-Powered Research Trend Intelligence Quadrant"
)

fig.add_vline(x=0, line_dash="dash")
fig.add_hline(y=0, line_dash="dash")

fig.update_layout(template="plotly_white")

fig.show()

In [64]:
impact_df = topics_df[(topics_df['slope_12m'] > 0)&
                    (topics_df["acceleration"] > 0)].copy()

impact_df = impact_df.sort_values('growth_score', ascending=False)

top10_impact = impact_df.head(10)

In [81]:
top10_impact.to_csv("top10_impact_topics.csv")

In [1]:
import pandas as pd
pd.read_csv("top10_impact_topics.csv", index_col=0)

,Count,Name,Representation,Representative_Docs,label,slope_12m,slope_24m,category,acceleration,growth_score,growth_size
Topic,,,,,,,,,,,
86,154,86_food_recipe_recipes_dietary,"['food', 'recipe', 'recipes', 'dietary', 'cook...",['Diffusion Model with Clustering-based Condit...,Recipe recipes dietary,0.267703,-0.008933,🌱 Emerging,0.276636,1.350140,50.000000
53,668,53_diffusion_sampling_generation_models,"['diffusion', 'sampling', 'generation', 'model...",['On Distillation of Guided Diffusion Models C...,Sampling generation models,0.104337,0.046023,🚀 Strong & Accelerating,0.058315,0.678797,43.076501
64,462,64_bandit_bandits_regret_multiarmed,"['bandit', 'bandits', 'regret', 'multiarmed', ...",['LLMs-augmented Contextual Bandit Contextual ...,Bandits regret multiarmed,0.063735,-0.002281,🌱 Emerging,0.066015,0.391186,40.110396
77,294,77_voting_auction_bidding_auctions,"['voting', 'auction', 'bidding', 'auctions', '...",['Online Fair Division for Personalized $2$-Va...,Auction bidding auctions,0.051395,0.001259,🚀 Strong & Accelerating,0.050136,0.292280,39.090396
57,584,57_personality_traits_social_roleplaying,"['personality', 'traits', 'social', 'roleplayi...",['Revealing Personality Traits: A New Benchmar...,Traits social roleplaying,0.043906,0.020873,🚀 Strong & Accelerating,0.023033,0.279750,38.961171
43,897,43_eeg_brain_signals_decoding,"['eeg', 'brain', 'signals', 'decoding', 'bci',...",['A Simple Review of EEG Foundation Models: Da...,Brain signals decoding,0.038972,-0.001932,🌱 Emerging,0.040905,0.265019,38.809252
51,677,51_privacy_private_dp_data,"['privacy', 'private', 'dp', 'data', 'differen...",['Privacy Preserving In-Context-Learning Frame...,Private dp data,0.025960,-0.000081,🌱 Emerging,0.026041,0.169234,37.821435
35,1010,35_causal_variables_causality_discovery,"['causal', 'variables', 'causality', 'discover...",['Data-Driven Causal Effect Estimation Based o...,Variables causality discovery,0.017025,-0.006983,🌱 Emerging,0.024009,0.117793,37.290930
32,1123,32_neural_pruning_networks_gradient,"['neural', 'pruning', 'networks', 'gradient', ...",['C-SWAP: Explainability-Aware Structured Prun...,Pruning networks gradient,0.013869,-0.002332,🌱 Emerging,0.016201,0.097426,37.080887
